# NPTEL Study Notes: Knowledge Graphs & LLMs

## 1. Foundations of Knowledge Graphs

A **Knowledge Graph (KG)** (often used interchangeably with **Knowledge Base (KB)**) is a structured representation of real-world facts comprising **entities** (nodes) connected by **relations** (directed, labeled edges).

### Graph Representation Models

#### Binary Relations

The elementary unit of factual knowledge in a KG is a **triple (or fact)**:


$$\tau = (s, r, o) \quad \text{or} \quad (h, r, t)$$

* $s$ or $h$: **Subject** / Head entity node
* $r$: **Relation type** / Predicate edge
* $o$ or $t$: **Object** / Tail entity node

$$\text{Example: } (\text{Barack Obama}, \text{born\_in}, \text{Honolulu})$$

#### Beyond Binary Relations ($N$-ary Relations)

Real-world facts frequently involve more than two entities or require qualification (temporal bounds, roles, provenance).

* **Representation:** Hyper-relational structures or reified relation nodes (often depicted as diamond-shaped intermediate nodes).
* **Examples:**
* **Ternary Movie Role Relation:** A table schema with `(Actor, Role, Movie)` cannot be captured by a single binary edge without losing semantic association. In graph form, a relation node connects `Tom Hanks`, `Jim Lovell` (Role), and `Apollo 13` (Film).
* **Attributed / Qualified Facts:** `(Barack Obama, president_of, USA)` qualified by attributes:

$$\text{start\_time} = 2009, \quad \text{end\_time} = 2017$$





```
       [Barack Obama] (Subject)
             |
             v
   <>--< president_of >--<> (Relation Node)
      |        |        |
      v        v        v
    [USA]   [2009]   [2017]
   (Object) (Start)  (End)

```

---

## 2. Anatomy of Production Knowledge Graphs: Wikidata

In production KGs such as **Wikidata**, strings are inherently ambiguous (e.g., *John Smith*, *David Carmel*). Unambiguous representation requires canonical, opaque identifiers.

| Component | Identifier Schema | Example | Attributes / Metadata |
| --- | --- | --- | --- |
| **Entity Node** | Prefixed with **`Q`** | `Q76` (*Barack Obama*), `Q12418` (*Mona Lisa*) | Canonical Label, Unique ID, Disambiguation Description (e.g., *"44th President of the USA"*), Aliases/Synonyms across languages (e.g., *Barry Obama*, *BHO*). |
| **Relation Edge** | Prefixed with **`P`** | `P39` (*position held*) | Canonical Relation Name, Unique ID, Multi-lingual labels, Associated Qualifiers (e.g., `start time`, `replaced by`, `elected by`). |

* **Multilingualism:** Production KGs store labels, aliases, and descriptions across hundreds of languages tied to the same canonical language-agnostic ID (`Q` or `P` code).

---

## 3. Structural Challenges in Knowledge Graphs

### KG Incompleteness & Completion (Link Prediction)

* **Open-World Assumption (OWA):** The absence of a triple does not imply it is false; it merely reflects missing information.
* **Empirical Scale of Incompleteness:** In **Freebase**, the place of birth is missing for **93.8%** of recorded person entities.
* **Knowledge Base Completion (KBC):** The algorithmic task of inferring missing triples $\tau = (h, r, t)$ from observed edges. It operates:
* **Explicitly:** Directly populating and materializing edges into the graph.
* **Implicitly:** Inferring missing facts on-the-fly during question-answering when consulted by an LLM.



### Cross-Lingual Entity and Relation Alignment

Aligning knowledge bases across languages (e.g., English KG $\mathcal{G}_{\text{en}}$ and Chinese KG $\mathcal{G}_{\text{zh}}$) requires identifying equivalent entities:


$$e_{\text{en}} \equiv e_{\text{zh}}$$

* **Surface Clues:** Surface-level translation and phonetic transliteration provide initial priors.
* **Structural/Topology Clues:** Two entities (e.g., *Casino Royale* and its Chinese counterpart) align reliably if their neighboring nodes (actors, directors, release years) and connecting relation types are also aligned.
* **Synergy:** Entity alignment, relation alignment, and graph completion form an interdependent, mutually reinforcing optimization loop.

```
 English KG:   [Daniel Craig] ----(actor_in)----> [Casino Royale]
                      |                                  |
               (Entity Alignment)                 (Entity Alignment)
                      v                                  v
 Chinese KG:   [丹尼尔·克雷格] ----(参演电影)----> [007: 大战皇家赌场]

```

---

## 4. Question Answering (QA): Semantic Interpretation vs. Differentiable KGs

Given a multi-hop factual question:


$$\text{"In what city is the Mona Lisa?"}$$

Answering requires combining:

1. **World Knowledge / Factual Storage:** $\text{Mona Lisa} \xrightarrow{\text{exhibited\_at}} \text{Louvre Museum} \xrightarrow{\text{located\_in}} \text{Paris}$.
2. **Type Checking & Spatial Inference:** $\text{Paris} \xrightarrow{\text{instance\_of}} \text{Capital/City}$; inferring that if an entity is inside a museum in a city, it is physically in that city.

---

### Architectural Paradigms

```
PARADIGM 1: SEMANTIC INTERPRETATION
Natural Query ---> [Semantic Parser] ---> Structured Query (SPARQL/SQL) ---> [KG Engine] ---> Output Answer
(Requires Schema Linking & Disambiguation; Modular & Fully Interpretable; Non-Differentiable)

PARADIGM 2: DIFFERENTIABLE (END-TO-END) KNOWLEDGE GRAPH
Query -------> [Dense Text Encoder (BERT)] ----\
                                                +---> [Cross-Attention Network] ---> Answer Subgraph
Graph Subset -> [Graph Neural Network (GNN)] ---/
(End-to-End Trainable; Soft Matching; Black-Box / Hard to Debug)

```

| Criterion | Semantic Interpretation (Symbolic) | Differentiable Knowledge Graph (Neural) |
| --- | --- | --- |
| **Pipeline** | Maps text $\to$ structured query (SPARQL/SQL) via semantic parsing. | Encodes text $\to$ dense vector; encodes graph $\to$ dense vectors; matches via neural attention. |
| **Encoders** | Grammar-based or sequence-to-tree/sequence-to-SQL neural parsers. | Text: BERT/BART; Graph: GNN, GCN, Graph Transformers. |
| **Execution** | Deterministic graph execution over explicit database engine. | Soft cross-attention scoring across entity/relation embedding matrices. |
| **Entity Linking** | Explicit disambiguation step required (maps text string to canonical ID, e.g., `"Mona Lisa"` $\to$ `Q12418`). | Implicitly handled through dense semantic vector proximity in latent space. |
| **Interpretability** | **High:** Explicit logical query can be inspected, debugged, and audited. | **Low:** Latent geometric/attention projections behave as black-box models. |
| **Trainability** | Hard to train end-to-end; errors in intermediate parsing cascade. | **Fully differentiable** and end-to-end trainable via backpropagation. |

---

## 5. Motivation for Combining LLMs with KGs

1. **Information Mutability & Real-Time Updates:**
* World states change dynamically (e.g., world leaders, sports champions).
* **LLM Limitation:** Knowledge is frozen in parametric weights. Updating requires computationally intensive fine-tuning or full pretraining.
* **KG Advantage:** Highly mutable, explicit storage. A node or edge can be updated or retracted in real time (within minutes on platforms like Wikipedia/Wikidata).


2. **Mitigating Hallucinations:**
* LLMs generate tokens via probabilistic decoding:

$$P(w_t \mid w_{<t})$$



This distribution can invent ungrounded, non-factual assertions.
* Grounding generation against explicit, verified KG triples bounds the decoding space to factual truths.


3. **Interpretability & Debuggability:**
* Diagnosing erroneous reasoning inside an opaque, billion-parameter LLM is challenging.
* Inspecting cross-attention weights over retrieved KG subgraphs explicitly highlights the factual premises the model retrieved and conditioned upon.


4. **The Head vs. Torso/Tail Distribution Problem:**
* LLMs perform well on popular "head" entities (e.g., *Barack Obama*, *Albert Einstein*) because they appear across millions of pretraining documents.
* For "torso" and "long-tail" entities (lesser-known individuals, regional cultural figures, low-resource domains), parametric recall drops off significantly. Here, structured external KG retrieval is essential.



---

## 6. Knowledge Graph Representation Paradigms

```
                          KG Representation Paradigms
                                       |
        +------------------------------+------------------------------+
        |                                                             |
Translation Models (TransE)                                Rotation Models (RotatE)
Points in vector space                                      Complex vector space
Relation = Spatial Translation                              Relation = Complex Rotation
h + r ≈ t                                                   h ∘ r ≈ t
        |                                                             |
        +------------------------------+------------------------------+
                                       |
                        Multiplicative / Factorization
                        Tensors: X ∈ R^(|E| x |R| x |E|)
                        Low-rank decomposition (e.g., RESCAL, DistMult)
                        Highest empirical expressivity

```

### 1. Translation-Based Models

* **Core Concept:** Entities are embedded as point vectors in continuous vector space $\mathbb{R}^d$.
* **Geometric Formulation:** A relation $r$ acts as a directional translation vector bridging head $h$ and tail $t$:

$$\mathbf{h} + \mathbf{r} \approx \mathbf{t}$$


* **Distance Scoring:**

$$f_r(h, t) = -\Vert{}\mathbf{h} + \mathbf{r} - \mathbf{t}\Vert{}_{1/2}$$



### 2. Rotation-Based Models

* **Core Concept:** Embeddings are mapped in the complex coordinate space $\mathbb{C}^d$.
* **Geometric Formulation:** The relation acts as an element-wise rotation (Hadamard product $\circ$) rotating the head vector to the tail:

$$\mathbf{t} = \mathbf{h} \circ \mathbf{r}, \quad \text{where } \vert{}r_i\vert{} = 1$$


* **Advantage:** Naturally models symmetric, anti-symmetric, and inversion relation patterns.

### 3. Factorization / Multiplicative Models

* **Core Concept:** Casts the knowledge graph as a third-order binary adjacency tensor:

$$\mathcal{X} \in \{0, 1\}^{\vert{}\mathcal{E}\vert{} \times \vert{}\mathcal{R}\vert{} \times \vert{}\mathcal{E}\vert{}}$$



where $\mathcal{X}_{ijk} = 1$ if triple $(e_i, r_j, e_k)$ exists, and $0$ otherwise.
* **Mechanism:** Decomposes the large, sparse tensor into low-rank entity and relation embedding matrices.
* **Score Function:** Bilinear product forms:

$$f_r(h, t) = \mathbf{h}^\top \mathbf{W}_r \mathbf{t}$$


* **Comparative Performance:** Achieves high empirical expressivity compared to basic translational models, though at the expense of higher parameter counts per relation.

---

## 7. Advanced Representation Challenges

### Modeling Type Hierarchies (Ontological Subsumption)

* Complex reasoning depends on taxonomic hypernymy:

$$\text{Camel} \subset \text{Mammal} \subset \text{Animal}$$


* Standard Euclidean embedding spaces (points and translations) struggle to model partial orders and hierarchical tree-like embeddings without distortion.
* **QA Impact:** Questions often filter on latent semantic types (e.g., *"Which scientist played violin?"* requires checking that $\text{Albert Einstein} \xrightarrow{\text{instance\_of}} \text{Scientist}$). If intermediate hypernym edges are unmaterialized, reasoning fails.

### Temporal Knowledge Graphs

Facts often hold valid only across specific temporal intervals $[t_{\text{start}}, t_{\text{end}}]$:


$$\tau = (s, r, o, [t_{\text{start}}, t_{\text{end}}])$$

* **Static Assumption Failure:** Treating relations as static produces contradictory facts (e.g., $(\text{Barack Obama}, \text{president\_of}, \text{USA})$ and $(\text{Donald Trump}, \text{president\_of}, \text{USA})$ cannot both be unconditionally true).
* **Temporal Reasoning:** Answering *"Who was the Prime Minister of Japan when the Shoemaker-Levy comet collided with Jupiter?"* requires:
1. Resolving the event's timestamp:

$$t_{\text{event}} = \text{July } 1994$$


2. Querying the temporal interval in the KG:

$$\text{PM}(x, \text{Japan}) \land (t_{\text{start}} \le 1994 \le t_{\text{end}})$$





---

## 8. High-Yield Summary (Quick Revision Matrix)

| Concept | Key Technical Takeaway | Common Exam Trap / Distinctions |
| --- | --- | --- |
| **KG Triple** | Canonical fact representation $(s, r, o)$. | Binary relations cannot natively represent $N$-ary relations without reification or hyper-edges. |
| **Wikidata IDs** | Entities = `Q` (e.g., `Q76`); Relations/Properties = `P` (e.g., `P39`). | IDs are completely opaque/meaningless; they prevent lexical ambiguity. |
| **KBC** | Predicting missing edges caused by the Open-World Assumption. | KBC is needed both *explicitly* (storage growth) and *implicitly* (online QA). |
| **Semantic Interpretation** | Translates question $\to$ SPARQL/SQL via parser. | Interpretable, but brittle and non-differentiable; requires explicit entity linking. |
| **Differentiable KG** | Encodes question & graph into dense vectors; matches via attention. | Fully end-to-end trainable; acts as a black box and is difficult to debug. |
| **LLM vs. KG** | LLMs provide language inference; KGs provide mutable, factual ground truth. | LLMs encode knowledge in static parameters; KGs allow real-time updates and prevent hallucinations. |
| **TransE vs. RotatE** | TransE uses vector addition ($\mathbf{h}+\mathbf{r}\approx\mathbf{t}$); RotatE uses complex rotations ($\mathbf{h}\circ\mathbf{r}\approx\mathbf{t}$). | Vector translations fail on symmetric relations; RotatE supports symmetry and inversion. |
| **Multiplicative Models** | Third-order tensor factorization over $\vert{}\mathcal{E}\vert{} \times \vert{}\mathcal{R}\vert{} \times \vert{}\mathcal{E}\vert{}$. | Often yields superior expressivity over simple distance-based translation models. |
| **Temporal KGs** | Adds time intervals $[t_{\text{start}}, t_{\text{end}}]$ to facts. | Static KGs cannot resolve time-dependent questions without explicit temporal qualifiers. |

---

## 9. Practice Assessment Questions

### Question 1 (Multiple Choice)

In the Wikidata architecture, which of the following represents an accurate structural distinction between entities and relations?

1. Entities are represented by human-readable string literals, whereas relations use alphanumeric identifiers.
2. Entities are assigned arbitrary unique identifiers prefixed with `Q`, whereas relations are assigned identifiers prefixed with `P`.
3. Entity IDs use `P` prefixes for primary nodes, and relation IDs use `Q` prefixes for property attributes.
4. Wikidata uses dense continuous vectors as primary identifiers to maintain end-to-end differentiability.

> **Answer:** **2**
> **Explanation:** Wikidata disambiguates entities using opaque numeric strings prefixed with `Q` (e.g., `Q76` for Barack Obama), while relations/properties are prefixed with `P` (e.g., `P39` for *position held*).

---

### Question 2 (Multiple Choice)

Why can an end-to-end differentiable Knowledge Graph Question-Answering system be more advantageous than a Semantic Interpretation system, and what is its primary operational drawback?

1. It supports logical verification; its drawback is that it requires manual SQL/SPARQL writing.
2. It eliminates the need for an intermediate formal grammar or logical form; its drawback is low interpretability and difficult debugging.
3. It guarantees 0% hallucination; its drawback is that it cannot process text queries using BERT.
4. It natively resolves $N$-ary relations; its drawback is an inability to compute cross-attention.

> **Answer:** **2**
> **Explanation:** Differentiable KGs use dense embeddings and cross-attention, avoiding rigid semantic parsers and brittle formal logical forms (SPARQL/SQL). However, because representations are continuous and latent, they function like black boxes, making error tracking and debugging challenging.

---

### Question 3 (Multiple Choice / Conceptual)

Consider a knowledge graph evaluated under the Open-World Assumption (OWA). The database lacks a triple linking entity `Albert Einstein` with the relation `plays_instrument` to `Violin`. What does the system conclude?

1. The fact is false; Einstein did not play the violin.
2. The fact is true by default using probabilistic graph smoothing.
3. The truth value is unknown; the absence of an edge indicates missing information, not falsity.
4. The system updates the entity ID to a null pointer.

> **Answer:** **3**
> **Explanation:** Under the Open-World Assumption, unobserved triples are treated as unknown rather than definitively false, which motivates Knowledge Base Completion (KBC).

---

### Question 4 (Numerical / Algorithmic Formulation)

A knowledge graph contains $\vert{}\mathcal{E}\vert{} = 50{,}000$ unique entities and $\vert{}\mathcal{R}\vert{} = 1{,}000$ distinct relations.

1. What are the dimensions of its full binary adjacency tensor $\mathcal{X}$?
2. If this tensor is modeled using a bilinear multiplicative model where each entity and relation is mapped to a latent embedding dimension of $d = 200$, what is the total number of embedding parameters required to represent the entities and relation matrices $\mathbf{W}_r \in \mathbb{R}^{d \times d}$?

> **Solution:**
> 1. The full 3D adjacency tensor $\mathcal{X}$ has dimensions:
> 
> $$\vert{}\mathcal{E}\vert{} \times \vert{}\mathcal{R}\vert{} \times \vert{}\mathcal{E}\vert{} = 50{,}000 \times 1{,}000 \times 50{,}000$$
> 
> 
> 2. Total parameters:
> * Entity embeddings:
> 
> $$\vert{}\mathcal{E}\vert{} \times d = 50{,}000 \times 200 = 10^7 \text{ parameters}$$
> 
> 
> * Relation matrices (for standard bilinear models where each relation has a matrix $\mathbf{W}_r \in \mathbb{R}^{d \times d}$):
> 
> $$\vert{}\mathcal{R}\vert{} \times d^2 = 1{,}000 \times (200 \times 200) = 1{,}000 \times 40{,}000 = 4 \times 10^7 \text{ parameters}$$
> 
> 
> * Total parameter footprint:
> 
> $$10^7 + 4 \times 10^7 = 5 \times 10^7 \text{ parameters (50 million parameters)}$$
> 
> 
> 
> 
> 
> 

---

### Question 5 (Short Explanatory / MCQ)

Which of the following scenarios best justifies augmenting an LLM with an external Knowledge Graph rather than relying solely on the LLM's parametric weights?

1. The question requires generating poetic, highly creative analogies about common concepts.
2. The question pertains to a low-resource regional political election that concluded yesterday.
3. The prompt asks for common English language grammatical corrections.
4. The task requires standard commonsense reasoning about everyday objects.

> **Answer:** **2**
> **Explanation:** LLMs struggle with recent updates (temporal lag) and long-tail/low-resource entities due to fixed parametric knowledge and training costs. KGs provide real-time updates and factual ground truth for non-head entities.